In [ ]:
import torch
import torch.nn as nn
from typing import Any, Dict, List
import matplotlib.pyplot as plt
import numpy as np

from src.models.student import PhisatNetEncoder, PhisatNetDecoder

## Architecture

## Setup

In [2]:
SEG_TASKS = {
    'lc': 11, 
    'anomaly_detection': 9, 
    'burned_area': 4, 
    'clouds': 2, 
    'fire': 3, 
    'worldfloods': 3
}

In [3]:
PATH_ENCODER = "/shared/projects/phisat2/bad_weights/encoder_sim_base.pth"
PATH_TASKS = {}
for task in SEG_TASKS.keys():
    PATH_TASKS[task] = f"/shared/projects/phisat2/bad_weights/decoder_{task}.pth"

In [4]:
# We put the encoder on GPU
encoder = PhisatNetEncoder(n_channels=8, base_filters=16, depth=3, channel_multipliers=[1,2,4,8]).to("cuda")
encoder.load_state_dict(torch.load(PATH_ENCODER, weights_only=True))
encoder.eval()

# We put the segmentation heads on CPU since they are small and we will move them one by one to GPU during inference
seg_heads = {}
for task_name, n_classes in SEG_TASKS.items():
    head = PhisatNetDecoder(n_classes=n_classes, base_filters=16, depth=3, channel_multipliers=[1,2,4,8])
    head.load_state_dict(torch.load(PATH_TASKS[task_name], weights_only=True), strict=False)
    head.eval()
    seg_heads[task_name] = head

## Inference

In [5]:
def multi_task_inference(images, active_tasks, encoder, seg_heads, device="cuda"):
    
    outputs = {task_name: None for task_name in seg_heads.keys()}
    
    encoder.eval()
    
    with torch.no_grad():
        
        features = encoder(images.to(device))
        
        for task in active_tasks:
            if task not in seg_heads:
                print(f"[WARNING] Task {task} ignored : not found in seg_heads.")
                continue
                
            head = seg_heads[task]
            head.eval()
            
            head.to(device)
            
            logits = head(features)
            
            outputs[task] = logits.cpu() 
            
            head.to("cpu")

    return outputs

## Tests

In [6]:
img_tensor = torch.randn(1, 8, 256, 256)  # Example input tensor
tasks_to_run = ['lc', 'clouds']  # Example active tasks

In [7]:
results = multi_task_inference(
     images=img_tensor, 
     active_tasks=tasks_to_run, 
     encoder=encoder, 
     seg_heads=seg_heads, 
     device="cuda"
 )

In [8]:
print("Landcover and Clouds:", results["lc"].shape, results["clouds"].shape)
print("Fire:", results["fire"])

Landcover and Clouds: torch.Size([1, 11, 256, 256]) torch.Size([1, 2, 256, 256])
Fire: None
